# Video Depth Anything — A Visual Tour of the Inference Pipeline

This notebook walks **3 RGB frames** through the complete **Video-Depth-Anything (ViT-S)** inference stack, printing tensor shapes and visualising every internal stage with matplotlib.

```
RGB Frames [T=3, H, W, 3]
  │
  ├─ 1. Preprocessing  (Resize → ImageNet-Normalise → CHW)
  │
  ├─ 2. DINOv2 ViT-S Encoder
  │     ├─ Patch Embedding  (14×14 Conv2d  →  37×37 = 1 369 tokens)
  │     ├─ CLS Token prepend
  │     ├─ Positional Encoding  (learned, shown as a 384-channel contactsheet)
  │     └─ 12 Transformer Blocks  (norm → attn → MLP)
  │           └─ 4 layers extracted: blocks 2, 5, 8, 11
  │
  └─ 3. DPTHeadTemporal
        ├─ Token → Spatial reshape
        ├─ Project layers  (1×1 Conv, channel reduction)
        ├─ Resize layers   (ConvTranspose ×4 / ×2 / Id / stride-2 Conv)
        ├─ TemporalModules [0,1]  on layer_3 & layer_4
        ├─ Scratch layers  (3×3 Conv → uniform 64-ch)
        ├─ RefineNet 4 → 3 → 2 → 1  (FeatureFusionBlocks, each shown step-by-step)
        ├─ TemporalModules [2,3]  on path_4 & path_3
        └─ Output Conv → Depth Map  [T, H, W]
```


In [ ]:
import sys, os, warnings, glob
sys.path.insert(0, '/home/user/Video-Depth-Anything')
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import cv2
from sklearn.decomposition import PCA
from collections import OrderedDict

plt.style.use('dark_background')
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': '#111111',
                     'axes.facecolor': '#111111', 'axes.edgecolor': '#444'})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")


In [ ]:
# ── helpers ───────────────────────────────────────────────────

def norm01(x):
    """Normalise ndarray/tensor to [0,1]."""
    if isinstance(x, torch.Tensor):
        x = x.detach().float().cpu().numpy()
    x = x.astype(np.float32)
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo + 1e-8)

def tinfo(t, label, indent=2):
    """Print shape + stats for a tensor."""
    if isinstance(t, torch.Tensor):
        a = t.detach().float().cpu().numpy()
    else:
        a = np.asarray(t, np.float32)
    pad = ' ' * indent
    print(f"{pad}{label:45s}  shape={list(a.shape)}"
          f"  min={a.min():.3f}  max={a.max():.3f}  mean={a.mean():.4f}")

def to_np(t):
    if isinstance(t, torch.Tensor):
        return t.detach().float().cpu().numpy()
    return np.asarray(t, np.float32)

def contactsheet_image(act, ncols=24):
    """
    Build a single tiled image of every channel activation map.
    act : [C, H, W] numpy float32
    Returns: [rows*H, cols*W] float32 in [0,1]
    """
    if isinstance(act, torch.Tensor):
        act = to_np(act)
    if act.ndim == 4:          # [N,C,H,W] – use frame 0
        act = act[0]
    C, H, W = act.shape
    nrows = int(np.ceil(C / ncols))
    sheet = np.zeros((nrows * H, ncols * W), dtype=np.float32)
    for c in range(C):
        r, col = divmod(c, ncols)
        ch = act[c].astype(np.float32)
        ch = norm01(ch)
        sheet[r*H:(r+1)*H, col*W:(col+1)*W] = ch
    return sheet

def show_contactsheet(act, title, ncols=24, cmap='viridis', figsize=None):
    """
    Display a full channel-activation contactsheet.
    act : [C,H,W] or [N,C,H,W]  tensor/ndarray
    """
    if isinstance(act, torch.Tensor):
        act = to_np(act)
    if act.ndim == 4:
        act = act[0]
    C, H, W = act.shape
    nrows = int(np.ceil(C / ncols))
    sheet = contactsheet_image(act, ncols)
    fw = max(12, min(ncols * W * 0.065, 26))
    fh = max(4,  min(nrows * H * 0.065, 20))
    if figsize:
        fw, fh = figsize
    fig, ax = plt.subplots(figsize=(fw, fh))
    im = ax.imshow(sheet, cmap=cmap, aspect='auto', interpolation='nearest')
    ax.set_title(
        f"{title}\n"
        f"{C} channels  |  each cell = {H}×{W} px  |  "
        f"grid = {nrows} rows × {ncols if C>=ncols else C} cols",
        fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
    plt.tight_layout()
    plt.show()

def show_row(arrays, titles, cmap=None, figsize=None, suptitle=None, vmin=None, vmax=None):
    """Show a horizontal strip of images."""
    n = len(arrays)
    fw = figsize[0] if figsize else 5*n
    fh = figsize[1] if figsize else 4
    fig, axes = plt.subplots(1, n, figsize=(fw, fh))
    if n == 1:
        axes = [axes]
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    for ax, arr, ttl in zip(axes, arrays, titles):
        arr = np.squeeze(to_np(arr))
        kw = {}
        if cmap:   kw['cmap'] = cmap
        if vmin is not None: kw['vmin'] = vmin
        if vmax is not None: kw['vmax'] = vmax
        ax.imshow(arr, **kw)
        ax.set_title(ttl, fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def show_heatmaps_per_frame(act, title, cmap='magma', frame_labels=None):
    """
    Show mean-magnitude heatmap per frame.
    act : [T, C, H, W]
    """
    act = to_np(act)
    T = act.shape[0]
    mean_map = np.abs(act).mean(axis=1)   # [T, H, W]
    labels = frame_labels or [f'Frame {i+1}' for i in range(T)]
    show_row(list(mean_map), labels, cmap=cmap, suptitle=title)

def unpack_temporal(t):
    """
    TemporalModule returns [B, C, T, H, W] (hook fires before the
    .permute(0,2,1,3,4).flatten(0,1) in DPTHeadTemporal.forward).
    Convert to [T, C, H, W] to match every other activation tensor.
    """
    a = to_np(t)
    if a.ndim == 5:  # [B, C, T, H, W]
        a = a.transpose(0, 2, 1, 3, 4)   # [B, T, C, H, W]
        a = a.reshape(-1, a.shape[2], a.shape[3], a.shape[4])  # [B*T, C, H, W]
    return a

print("Helpers defined.")


## 1 — Load Model
Using **ViT-Small** (embed_dim=384, 12 blocks, 28M params). Checkpoints are downloaded automatically from HuggingFace if not already present locally.

In [ ]:
from video_depth_anything.video_depth import VideoDepthAnything

ENCODER = 'vits'
CFG = {'vits': dict(encoder='vits', features=64,  out_channels=[48,  96,  192, 384]),
       'vitb': dict(encoder='vitb', features=128, out_channels=[96,  192, 384, 768]),
       'vitl': dict(encoder='vitl', features=256, out_channels=[256, 512, 1024, 1024])}

model = VideoDepthAnything(**CFG[ENCODER])

# ── Try local checkpoints first ────────────────────────────
CKPT_NAME = 'Video_Depth_Anything_Small.pth'
local_candidates = [
    f'./checkpoints/{CKPT_NAME}',
    f'/home/user/Video-Depth-Anything/checkpoints/{CKPT_NAME}',
]
loaded = False
for p in local_candidates:
    if os.path.exists(p):
        sd = torch.load(p, map_location='cpu')
        model.load_state_dict(sd)
        print(f"Loaded checkpoint: {p}")
        loaded = True
        break

if not loaded:
    try:
        from huggingface_hub import hf_hub_download
        os.makedirs('./checkpoints', exist_ok=True)
        p = hf_hub_download(repo_id='depth-anything/Video-Depth-Anything',
                            filename=CKPT_NAME, local_dir='./checkpoints')
        sd = torch.load(p, map_location='cpu')
        model.load_state_dict(sd)
        print(f"Downloaded checkpoint: {p}")
        loaded = True
    except Exception as e:
        print(f"WARNING – checkpoint not loaded ({e}). Random weights used.")

model = model.to(device).eval()
enc = model.pretrained   # DinoVisionTransformer

print(f"\n{'─'*60}")
print(f"  Encoder          : DINOv2 ViT-Small")
print(f"  embed_dim        : {enc.embed_dim}")
print(f"  n_blocks         : {enc.n_blocks}")
print(f"  num_heads        : {enc.num_heads}")
print(f"  patch_size       : {enc.patch_size}")
print(f"  pos_embed shape  : {list(enc.pos_embed.shape)}")
print(f"  Extracted blocks : {model.intermediate_layer_idx[ENCODER]}")
print(f"  DPT features     : {CFG[ENCODER]['features']}")
print(f"  DPT out_channels : {CFG[ENCODER]['out_channels']}")
print(f"  Temporal modules : 4  (2 on feature maps, 2 on decoder paths)")
print(f"{'─'*60}")


## 2 — Test Frames
Three synthetic frames: a gradient background with a circle (moves left→right across frames), a rectangle, and a small distant circle — useful depth cues.

In [ ]:
def make_frame(idx, H=480, W=640):
    """Synthetic RGB frame for testing."""
    frame = np.zeros((H, W, 3), dtype=np.uint8)
    xx, yy = np.meshgrid(np.linspace(0,1,W), np.linspace(0,1,H))
    frame[:,:,0] = (50 + 120*xx*(0.7+0.3*idx/2)).clip(0,255).astype(np.uint8)
    frame[:,:,1] = (30  + 80*yy).clip(0,255).astype(np.uint8)
    frame[:,:,2] = (120 - 70*xx + 30*idx/2).clip(0,255).astype(np.uint8)
    # Foreground circle – moves right as idx increases
    cx, cy = int(W*(0.22 + 0.22*idx/2)), int(H*0.45)
    cv2.circle(frame, (cx, cy), 90, (255, 210, 60), -1)
    cv2.circle(frame, (cx, cy), 90, (200, 150, 20),  4)
    # Stationary rectangle (mid-ground)
    cv2.rectangle(frame, (W//2-70, H//2-45), (W//2+70, H//2+45), (80, 200, 255), -1)
    cv2.rectangle(frame, (W//2-70, H//2-45), (W//2+70, H//2+45), (40, 130, 200), 3)
    # Small far circle (background)
    cv2.circle(frame, (int(W*0.82), int(H*0.22)), 38, (220, 100, 160), -1)
    return frame

RAW = [make_frame(i) for i in range(3)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Input RGB Frames  (3 × 480×640×3  uint8)', fontsize=12)
for i, (ax, f) in enumerate(zip(axes, RAW)):
    ax.imshow(f)
    ax.set_title(f'Frame {i+1}', fontsize=10)
    ax.axis('off')
plt.tight_layout(); plt.show()

for i, f in enumerate(RAW):
    tinfo(f, f'raw_frame[{i}]')


## 3 — Preprocessing Pipeline

Three steps before the encoder sees the data:

| Step | Class | What it does |
|------|-------|-------------|
| **Resize** | `Resize` | Bicubic resize so the *smaller* side ≥ 518, output must be a multiple of **14** (patch size) |
| **Normalise** | `NormalizeImage` | Subtracts ImageNet mean, divides by std (values move from [0,1] to ~[−2,+2]) |
| **PrepareForNet** | `PrepareForNet` | Transposes HWC → CHW, ensures float32 C-contiguous memory |


In [ ]:
from video_depth_anything.util.transform import Resize, NormalizeImage, PrepareForNet
from torchvision.transforms import Compose

INPUT_SIZE = 518   # smaller edge target; must be multiple of 14

# ── Build each step as its own transform so we can inspect intermediates ─────
resize_tf = Resize(
    width=INPUT_SIZE, height=INPUT_SIZE, resize_target=False,
    keep_aspect_ratio=True, ensure_multiple_of=14,
    resize_method='lower_bound', image_interpolation_method=cv2.INTER_CUBIC)
norm_tf   = NormalizeImage(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
prep_tf   = PrepareForNet()

def preprocess_step_by_step(frame_uint8):
    """Return (float01_HWC, resized_HWC, normalised_HWC, chw_tensor)."""
    f01   = frame_uint8.astype(np.float32) / 255.0
    res   = resize_tf({'image': f01.copy()})['image']
    nor   = norm_tf({'image': res.copy()})['image']
    chw   = prep_tf({'image': nor.copy()})['image']   # → [3, H, W] float32
    return f01, res, nor, chw

steps = [preprocess_step_by_step(f) for f in RAW]
f01s, resized, normed, chwlist = zip(*steps)

print("Shapes at each preprocessing stage:")
tinfo(RAW[0],         'raw uint8   [H,W,3]')
tinfo(resized[0],     'after Resize        ')
tinfo(normed[0],      'after Normalise     ')
tinfo(chwlist[0],     'after PrepareForNet ')

# ── Visual comparison ────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
fig.suptitle('Preprocessing Pipeline — one row per frame', fontsize=12)
col_titles = ['Original (uint8)', 'After Resize (float)', 'After Normalise (shifted for display)', 'Per-channel (R/G/B) normalised']
for c, ct in enumerate(col_titles):
    axes[0, c].set_title(ct, fontsize=8, color='#aaa')

for row in range(3):
    axes[row, 0].imshow(RAW[row])
    axes[row, 1].imshow(resized[row])
    # Normalised: shift back to [0,1] for display
    axes[row, 2].imshow(norm01(normed[row]))
    # Per-channel heatmaps of the normalised array
    chw = chwlist[row]    # [3, H, W]
    rgb_side = np.stack([norm01(chw[c]) for c in range(3)], axis=2)
    axes[row, 3].imshow(rgb_side)
    for ax in axes[row]: ax.axis('off')

plt.tight_layout(); plt.show()

# ── Pixel distribution before / after normalisation ──────────
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
fig.suptitle('ImageNet Normalisation: pixel value distribution (frame 0)', fontsize=11)
cnames = ['Red', 'Green', 'Blue']
colours = ['#ff6666', '#66ff88', '#6699ff']
for c in range(3):
    axes[c].hist(resized[0][:,:,c].flatten(), bins=60, alpha=0.6,
                 color=colours[c], label='Before (0–1)', density=True)
    axes[c].hist(chwlist[0][c].flatten(), bins=60, alpha=0.5,
                 color='white',   label='After (normalised)', density=True)
    axes[c].set_title(cnames[c], fontsize=9)
    axes[c].legend(fontsize=7)
plt.tight_layout(); plt.show()

# ── Assemble tensor for the model: [B=1, T=3, C, H, W] ──────
frames_chw  = np.stack(chwlist)                             # [3, 3, H, W]
INPUT_TENSOR = torch.from_numpy(frames_chw).unsqueeze(0).to(device)  # [1,3,3,H,W]
B, T, C, H_in, W_in = INPUT_TENSOR.shape
patch_h = H_in // 14
patch_w = W_in // 14
print(f"\nModel input tensor : {list(INPUT_TENSOR.shape)}  [B, T, C, H, W]")
print(f"Spatial patch grid : {patch_h} × {patch_w} = {patch_h*patch_w} patches")


## 4 — Patch Embedding

The first thing the encoder does is split each frame into non-overlapping **14×14 patches** and linearly project each patch to a 384-dim vector using a single `Conv2d(3, 384, kernel_size=14, stride=14)`.

For a 518×518 frame: 518/14 = **37 patches per side** → **1 369 patch tokens** per frame.


In [ ]:
# ── Visualise the 14×14 patch grid on frame 1 ────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Patch Embedding — 14×14 grid  ({H_in}×{W_in} input)', fontsize=11)

ref_frame = norm01(chwlist[0]).transpose(1,2,0)   # [H,W,3] for display
axes[0].imshow(ref_frame)
axes[0].set_title(f'Frame 1 — {H_in}×{W_in}  ({patch_h}×{patch_w} patch grid)', fontsize=9)
# Draw patch grid lines
for y in range(0, H_in, 14):
    axes[0].axhline(y-0.5, color='#ff6', linewidth=0.3, alpha=0.5)
for x in range(0, W_in, 14):
    axes[0].axvline(x-0.5, color='#ff6', linewidth=0.3, alpha=0.5)
# Highlight top-left 3×3 patch block
rect = mpatches.Rectangle((0, 0), 42, 42, linewidth=2, edgecolor='cyan', facecolor='none')
axes[0].add_patch(rect)
axes[0].axis('off')

# ── Visualise the first 36 patch-embed projection kernels ─────
# patch_embed.proj is Conv2d(3,384,14,14) – kernel [384,3,14,14]
kern = to_np(enc.patch_embed.proj.weight)   # [384, 3, 14, 14]
# Build a small contactsheet of first 36 kernels (RGB channels averaged)
nk = 36
ncols_k = 9
nrows_k  = nk // ncols_k
ksheet = np.zeros((nrows_k*14, ncols_k*14, 3))
for ki in range(nk):
    r, c = divmod(ki, ncols_k)
    for ch in range(3):
        ksheet[r*14:(r+1)*14, c*14:(c+1)*14, ch] = norm01(kern[ki, ch])
axes[1].imshow(ksheet)
axes[1].set_title(f'First {nk} patch-embed projection kernels (each 14×14, RGB)', fontsize=9)
# Draw kernel boundaries
for y in range(0, nrows_k*14, 14):
    axes[1].axhline(y-0.5, color='#888', lw=0.5)
for x in range(0, ncols_k*14, 14):
    axes[1].axvline(x-0.5, color='#888', lw=0.5)
axes[1].axis('off')
plt.tight_layout(); plt.show()

# ── Forward through patch_embed only ─────────────────────────
x_flat = INPUT_TENSOR.flatten(0,1)   # [T=3, C, H, W]
with torch.no_grad():
    patch_tokens = enc.patch_embed(x_flat)   # [T, N, D]

tinfo(x_flat,        'Input to patch_embed  [T, C, H, W]')
tinfo(patch_tokens,  'Output patch tokens   [T, N, D]    ')
print(f"  → Each of {T} frames → {patch_tokens.shape[1]} tokens of dim {patch_tokens.shape[2]}")

# ── Show spatial mean of patch embeddings as a heatmap ────────
pt_spatial = to_np(patch_tokens).reshape(T, patch_h, patch_w, -1)  # [T, pH, pW, D]
pt_mean    = np.abs(pt_spatial).mean(-1)   # [T, pH, pW]
show_row(list(pt_mean),
         [f'Frame {i+1}: mean |activation| across {pt_spatial.shape[-1]} dims' for i in range(T)],
         cmap='magma', suptitle='Patch Embedding — spatial mean activation magnitude')


## 5 — Positional Encoding

DINOv2 uses a **learned** positional embedding: `pos_embed` is a parameter of shape `[1, 1370, 384]` where position 0 is reserved for the CLS token and positions 1–1369 map to the 37×37 patch grid.

We visualise two things:
1. **The raw positional embedding** as a contactsheet — for each of the 384 dimensions, what spatial pattern does it encode?
2. **PCA to 3 components** projected as RGB — a single image revealing the overall spatial structure learned by the positional encoding.


In [ ]:
# ── Extract patch positional embeddings ──────────────────────
pos_embed_full = to_np(enc.pos_embed[0])   # [1370, 384]  (CLS + patches)
cls_pe  = pos_embed_full[0:1]              # [1, 384]
patch_pe = pos_embed_full[1:]              # [1369, 384]

tinfo(pos_embed_full, 'pos_embed (full)  [1+N, D]')
tinfo(patch_pe,       'patch pos_embed   [N, D]  ')

# ── Stored pos_embed uses the *native* 518×518 training grid (always 37×37).
# For non-square inputs DINOv2 bicubic-interpolates it at runtime.
# We visualise the stored parameter, so use its own grid shape – not patch_h/patch_w.
N_pe = enc.pos_embed.shape[1] - 1          # 1369  (cls excluded)
pe_h = pe_w = int(round(N_pe ** 0.5))      # 37  (square by construction for ViT-S)
print(f"Stored PE grid  : {pe_h}×{pe_w} = {N_pe} patches  (native 518×518 training resolution)")
print(f"Actual input grid: {patch_h}×{patch_w} = {patch_h*patch_w} patches  "
      f"(pos_embed is bicubic-interpolated at runtime for non-square / different-size inputs)")

# Reshape to spatial using the stored PE dimensions
pe_spatial = patch_pe.reshape(pe_h, pe_w, enc.embed_dim)  # [37, 37, 384]
print(f"Reshaped to spatial: {pe_spatial.shape}  [pe_h, pe_w, D]")

# ── 1. Contactsheet: all 384 positional encoding channels ─────
pe_for_cs = pe_spatial.transpose(2, 0, 1)   # [384, 37, 37]
show_contactsheet(pe_for_cs,
    title='Positional Encoding — learned pos_embed (ViT-S, native 37×37 grid)',
    ncols=24, cmap='RdBu_r', figsize=(20, 13))

# ── 2. PCA → 3 components as RGB ─────────────────────────────
pca = PCA(n_components=3)
pe_flat  = patch_pe   # [1369, 384]
pe_pca   = pca.fit_transform(pe_flat)   # [1369, 3]
pe_pca_img = pe_pca.reshape(pe_h, pe_w, 3)   # use stored PE grid
pe_rgb = norm01(pe_pca_img)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('Positional Encoding — PCA visualisation (native 37×37 stored grid)', fontsize=11)
axes[0].imshow(pe_rgb)
axes[0].set_title('PCA top-3 as RGB  (spatial structure)', fontsize=9)
axes[0].axis('off')

# Show each PC separately
for pc in range(2):
    comp = pe_pca.reshape(pe_h, pe_w, 3)[:,:,pc]
    im = axes[pc+1].imshow(norm01(comp), cmap='RdBu_r')
    axes[pc+1].set_title(f'PC {pc+1}  (explains {pca.explained_variance_ratio_[pc]*100:.1f}% var)', fontsize=9)
    axes[pc+1].axis('off')
    plt.colorbar(im, ax=axes[pc+1], fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

# ── 3. Show the positional encoding ADDED to patch tokens ─────
with torch.no_grad():
    tokens_with_pe = enc.prepare_tokens_with_masks(x_flat, None)  # [T, 1+N, D]

tinfo(tokens_with_pe, 'tokens after CLS + pos_embed  [T, 1+N, D]')

# Visualise the delta: tokens_with_pe (patch part) vs raw patch_tokens
tok_np = to_np(tokens_with_pe[:, 1:, :])   # [T, N, D]
delta  = tok_np - to_np(patch_tokens)       # positional encoding contribution

print("\nMean absolute contribution of positional encoding:")
for i in range(T):
    print(f"  Frame {i+1}: mean|Δ|={np.abs(delta[i]).mean():.4f}  "
          f"(token mean|val|={np.abs(tok_np[i]).mean():.4f})")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Positional Encoding — per-frame mean contribution |Δ| across dimensions', fontsize=10)
for i in range(T):
    hm = np.abs(delta[i]).mean(-1).reshape(patch_h, patch_w)
    im = axes[i].imshow(hm, cmap='hot')
    axes[i].set_title(f'Frame {i+1}: mean |pos_enc| per patch', fontsize=8)
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()


## 6 — Register Hooks & Run the Full Forward Pass

We attach `register_forward_hook` to every module we want to inspect, run a single forward pass, and store all activations in `A`. The rest of the notebook visualises them in order.


In [ ]:
A = {}   # activation store: name → tensor

hooks = []
def hook(name):
    def _h(module, inp, out):
        if isinstance(out, tuple):
            A[name] = out[0].detach().clone() if isinstance(out[0], torch.Tensor) else None
        else:
            A[name] = out.detach().clone()
    return _h

def inp_hook(name, arg_idx=0):
    """Hook that captures an *input* rather than an output."""
    def _h(module, inp, out):
        if arg_idx < len(inp) and isinstance(inp[arg_idx], torch.Tensor):
            A[name] = inp[arg_idx].detach().clone()
    return _h

# ── DINOv2 encoder ─────────────────────────────────────────
hooks.append(enc.patch_embed.register_forward_hook(hook('pe_out')))
for i, blk in enumerate(enc.blocks):
    hooks.append(blk.register_forward_hook(hook(f'block_{i}')))

# ── DPT head – feature projection & resize ────────────────
head = model.head
for i in range(4):
    hooks.append(head.projects[i].register_forward_hook(hook(f'proj_{i}')))
    hooks.append(head.resize_layers[i].register_forward_hook(hook(f'resize_{i}')))

# ── Temporal modules ───────────────────────────────────────
for i, mm in enumerate(head.motion_modules):
    hooks.append(mm.register_forward_hook(hook(f'temporal_{i}')))

# ── Scratch layers ─────────────────────────────────────────
hooks.append(head.scratch.layer1_rn.register_forward_hook(hook('scratch_1')))
hooks.append(head.scratch.layer2_rn.register_forward_hook(hook('scratch_2')))
hooks.append(head.scratch.layer3_rn.register_forward_hook(hook('scratch_3')))
hooks.append(head.scratch.layer4_rn.register_forward_hook(hook('scratch_4')))

# ── RefineNet internals ────────────────────────────────────
for rn_name in ['refinenet4', 'refinenet3', 'refinenet2', 'refinenet1']:
    rn = getattr(head.scratch, rn_name)
    hooks.append(rn.register_forward_hook(hook(rn_name)))
    hooks.append(rn.resConfUnit1.register_forward_hook(hook(f'{rn_name}.rcu1')))
    hooks.append(rn.resConfUnit2.register_forward_hook(hook(f'{rn_name}.rcu2')))
    hooks.append(rn.out_conv.register_forward_hook(hook(f'{rn_name}.out_conv')))
    # Inside ResidualConvUnit
    hooks.append(rn.resConfUnit1.conv1.register_forward_hook(hook(f'{rn_name}.rcu1.conv1')))
    hooks.append(rn.resConfUnit1.conv2.register_forward_hook(hook(f'{rn_name}.rcu1.conv2')))
    hooks.append(rn.resConfUnit2.conv1.register_forward_hook(hook(f'{rn_name}.rcu2.conv1')))
    hooks.append(rn.resConfUnit2.conv2.register_forward_hook(hook(f'{rn_name}.rcu2.conv2')))

# ── Output convolutions ────────────────────────────────────
hooks.append(head.scratch.output_conv1.register_forward_hook(hook('out_conv1')))
hooks.append(head.scratch.output_conv2.register_forward_hook(hook('out_conv2')))

# ── Run forward pass ───────────────────────────────────────
with torch.no_grad():
    DEPTH_OUT = model(INPUT_TENSOR)   # [B, T, H, W]

# Remove hooks
for h in hooks:
    h.remove()

print("Forward pass complete.  Keys captured:")
for k, v in A.items():
    if v is not None:
        tinfo(v, k)


## 7 — DINOv2 Transformer Blocks (1–12)

Each block output has shape `[T, 1+N, D]` (CLS token at index 0 + N=1369 patch tokens, D=384).

We visualise the **mean absolute activation magnitude** across the 384 channels, arranged on the 37×37 spatial grid — one heatmap per block. The 4 extracted layers (blocks 2, 5, 8, 11) are highlighted.

Block structure (per block): `x = x + ls1(attn(norm1(x)))` then `x = x + ls2(mlp(norm2(x)))`


In [ ]:
EXTRACT_BLOCKS = set(model.intermediate_layer_idx[ENCODER])   # {2, 5, 8, 11}
n_blocks       = enc.n_blocks                                  # 12

# Build one heatmap per block (frame 1, patch tokens, mean|act| across D)
block_maps = []
for i in range(n_blocks):
    blk_out = to_np(A[f'block_{i}'])   # [T, 1+N, D]
    patches  = blk_out[:, 1:, :]       # remove CLS → [T, N, D]
    mean_abs = np.abs(patches).mean(-1)  # [T, N]
    hm = mean_abs[0].reshape(patch_h, patch_w)   # frame 0 as representative
    block_maps.append(norm01(hm))

fig, axes = plt.subplots(3, 4, figsize=(17, 12))
fig.suptitle('DINOv2 Blocks 0–11 — mean |activation| per patch (frame 1)\n'
             'cyan border = extraction point fed to DPT head', fontsize=11)
for i, (ax, hm) in enumerate(zip(axes.flatten(), block_maps)):
    im = ax.imshow(hm, cmap='inferno', vmin=0, vmax=1)
    border_col = '#00ffff' if i in EXTRACT_BLOCKS else '#444'
    for spine in ax.spines.values():
        spine.set_edgecolor(border_col)
        spine.set_linewidth(3 if i in EXTRACT_BLOCKS else 0.5)
    tag = ' ← extracted' if i in EXTRACT_BLOCKS else ''
    ax.set_title(f'Block {i}{tag}', fontsize=8,
                 color='cyan' if i in EXTRACT_BLOCKS else 'white')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# ── Plot token norm evolution (CLS + mean patch) ──────────────
cls_norms   = []
patch_norms = []
for i in range(n_blocks):
    b = to_np(A[f'block_{i}'])   # [T, 1+N, D]
    cls_norms.append(  np.linalg.norm(b[:, 0,  :], axis=-1).mean() )
    patch_norms.append(np.linalg.norm(b[:, 1:, :], axis=-1).mean() )

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(cls_norms,   marker='o', label='CLS token L2 norm', color='#ff9f40')
ax.plot(patch_norms, marker='s', label='Mean patch token L2 norm', color='#4fc3f7')
for bi in EXTRACT_BLOCKS:
    ax.axvline(bi, color='#00ffcc', linewidth=1.5, linestyle='--', alpha=0.7)
ax.set_xlabel('Block index'); ax.set_ylabel('Mean L2 norm')
ax.set_title('Token norm evolution across 12 blocks  (dashed cyan = extraction points)', fontsize=10)
ax.legend(); ax.set_xticks(range(n_blocks))
plt.tight_layout(); plt.show()


## 8 — The 4 Extracted Feature Layers (Contactsheets)

After blocks **2, 5, 8, 11** the DINOv2 encoder is tapped. Each extracted tensor is `[T, N, D] = [3, 1369, 384]`. Here we reshape the N patch tokens back to the 37×37 spatial grid and show **all 384 activation channels** as a contactsheet (16 rows × 24 cols of 37×37 maps).

Each cell in the contactsheet corresponds to one of the 384 feature dimensions, arranged spatially — you can read off which patches activate each feature.


In [ ]:
BLOCK_INDICES = model.intermediate_layer_idx[ENCODER]   # [2, 5, 8, 11]

for bi in BLOCK_INDICES:
    feat = to_np(A[f'block_{bi}'])   # [T, 1+N, D]
    # Remove CLS, reshape to spatial
    patches = feat[:, 1:, :]                           # [T, N, D]
    spatial = patches[0].reshape(patch_h, patch_w, -1) # [pH, pW, D]  frame 0
    cs_input = spatial.transpose(2, 0, 1)              # [D, pH, pW]

    tinfo(torch.from_numpy(feat), f'block_{bi} output [T, 1+N, D]')
    show_contactsheet(
        cs_input,
        title=f'Block {bi} — all {enc.embed_dim} activation channels  (frame 1, {patch_h}×{patch_w} patches)',
        ncols=24, cmap='viridis', figsize=(22, 15))


## 9 — DPTHeadTemporal: Overview

The DPT (Dense Prediction Transformer) head decodes the 4 extracted feature layers into a dense depth map. It has several stages:

```
4 layer features  [T, 1369, 384]  each
  │
  ├─ (use_clstoken=False, so no readout projection needed)
  │
  ├─ Reshape: [T, 1369, 384]  →  [T, 384, 37, 37]
  │
  ├─ projects[i]    1×1 Conv: 384 → [48 / 96 / 192 / 384] channels
  │
  ├─ resize_layers[i]
  │     layer 0: ConvTranspose2d ×4  →  [48,  148, 148]
  │     layer 1: ConvTranspose2d ×2  →  [96,   74,  74]
  │     layer 2: Identity            →  [192,  37,  37]
  │     layer 3: Conv2d stride=2     →  [384,  19,  19]
  │
  ├─ motion_modules[0] on layer_3, motion_modules[1] on layer_4   (temporal attention)
  │
  ├─ scratch.layer{1-4}_rn  3×3 Conv → uniform 64 channels
  │     layer_1_rn: [48, 148,148] → [64, 148,148]
  │     layer_2_rn: [96,  74, 74] → [64,  74, 74]
  │     layer_3_rn: [192, 37, 37] → [64,  37, 37]
  │     layer_4_rn: [384, 19, 19] → [64,  19, 19]
  │
  ├─ refinenet4  layer_4_rn → path_4  [64, 37, 37]  (upsample to layer_3 size)
  ├─ motion_modules[2] on path_4
  ├─ refinenet3  (path_4, layer_3_rn) → path_3  [64, 74, 74]
  ├─ motion_modules[3] on path_3
  ├─ refinenet2  (path_3, layer_2_rn) → path_2  [64, 148, 148]
  ├─ refinenet1  (path_2, layer_1_rn) → path_1  [64, 296, 296]  (×2 upsample)
  │
  ├─ output_conv1  Conv2d 64→32  →  [32, 296, 296]
  ├─ interpolate to (518, 518)   →  [32, 518, 518]
  └─ output_conv2  32→32→1       →  [1,  518, 518]
```


### 9a — Token→Spatial Reshape then 1×1 Project Layers

Each extracted layer goes from `[T, N, D]` → `[T, D, pH, pW]` (permute+reshape), then a `1×1 Conv2d` projects D channels to the target channel count.

In [ ]:
print("Project layers:  projects[i] are 1×1 Conv2d  (D → out_channels[i])")
print(f"  out_channels = {CFG[ENCODER]['out_channels']}")

ch_labels   = ['48', '96', '192', '384']
resize_desc = ['ConvTranspose2d ×4 → 148×148',
               'ConvTranspose2d ×2 → 74×74',
               'Identity         → 37×37',
               'Conv2d stride=2  → 19×19']

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle('Project layers  (proj[i])  then  resize_layers[i] — frame 1', fontsize=11)
row_titles = ['After projects[i]  (1×1 Conv, spatial=37×37)',
              'After resize_layers[i]',
              'Difference in spatial size (mean |act|)']

for i in range(4):
    pj  = to_np(A[f'proj_{i}'])    # [T, C_out, 37, 37]
    rz  = to_np(A[f'resize_{i}'])  # [T, C_out, H', W']
    tinfo(torch.from_numpy(pj),  f'proj_{i}  ')
    tinfo(torch.from_numpy(rz),  f'resize_{i}')

    mean_pj = np.abs(pj[0]).mean(0)   # [37, 37]
    mean_rz = np.abs(rz[0]).mean(0)   # [H', W']

    axes[0, i].imshow(norm01(mean_pj), cmap='magma')
    axes[0, i].set_title(f'layer {i+1}: {ch_labels[i]} ch, 37×37', fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(norm01(mean_rz), cmap='magma')
    axes[1, i].set_title(f'{resize_desc[i]}', fontsize=7)
    axes[1, i].axis('off')

    # Contactsheet of first 16 channels after resize (to see individual features)
    rz_cs = make_contactsheet(rz[0][:16], ncols=8)
    axes[2, i].imshow(rz_cs, cmap='viridis', aspect='auto')
    axes[2, i].set_title(f'First 16 channels (contactsheet)', fontsize=7)
    axes[2, i].axis('off')

for r in range(3):
    axes[r, 0].set_ylabel(row_titles[r], fontsize=8)
plt.tight_layout(); plt.show()

# ── Full contactsheets for resize layer outputs ───────────────
for i in range(4):
    rz = to_np(A[f'resize_{i}'])
    show_contactsheet(rz[0], ncols=min(rz.shape[1], 24),
        title=f'resize_layers[{i}]  output — {rz.shape[1]} channels  {rz.shape[2]}×{rz.shape[3]}',
        cmap='viridis', figsize=(16, 10))


### 9b — Temporal Modules (motion_modules[0] and [1])

Two `TemporalModule` instances add **cross-frame temporal attention** on layer_3 and layer_4 before the scratch layers.

**Internal flow:**
```
input  [B, C, F, H, W]   (B=1, F=T frames)
  → GroupNorm
  → Linear project (C → inner_dim)
  → Rearrange to [B*H*W, F, inner_dim]   (spatial tokens attend over time)
  → N × TemporalTransformerBlock
       ├─ LayerNorm
       ├─ TemporalAttention  (self-attn over F=3 time steps)
       │    ├─ + sinusoidal positional encoding over frames
       │    └─ scaled dot-product attention [F, F]
       └─ FeedForward (GEGLU)
  → Linear project back (inner_dim → C)
  → + residual
output [B, C, F, H, W]
```
`zero_initialize=True` means the temporal branch starts as a zero perturbation and gradually activates during training — at inference the residual connection is the dominant path.


In [ ]:
for mm_idx in [0, 1]:
    # The temporal module sees layer_3 or layer_4 reshaped to [B, C, T, H, W]
    # Our hook captures the output of the full TemporalModule
    tm_out = unpack_temporal(A[f'temporal_{mm_idx}'])   # [B,C,T,H,W] → [T,C,H,W]
    resize_in = to_np(A[f'resize_{2+mm_idx}'])  # input to temporal (layer_3 / layer_4)

    tinfo(torch.from_numpy(resize_in), f'temporal[{mm_idx}] input  (resize_{2+mm_idx})')
    tinfo(torch.from_numpy(tm_out),    f'temporal[{mm_idx}] output')

    delta = tm_out - resize_in
    print(f"  Temporal residual: mean|Δ|={np.abs(delta).mean():.5f}  "
          f"max|Δ|={np.abs(delta).max():.5f}")

    # Visualise: for each frame show mean|act| before and after temporal
    fig, axes = plt.subplots(2, T, figsize=(12, 6))
    fig.suptitle(f'Temporal Module [{mm_idx}]  '
                 f'({"layer_3" if mm_idx==0 else "layer_4"})  '
                 f'— mean |activation| per spatial position', fontsize=10)
    for fr in range(T):
        im0 = axes[0, fr].imshow(norm01(np.abs(resize_in[fr]).mean(0)), cmap='magma')
        axes[0, fr].set_title(f'Frame {fr+1} — INPUT', fontsize=8)
        axes[0, fr].axis('off')
        im1 = axes[1, fr].imshow(norm01(np.abs(tm_out[fr]).mean(0)), cmap='magma')
        axes[1, fr].set_title(f'Frame {fr+1} — OUTPUT', fontsize=8)
        axes[1, fr].axis('off')
    plt.colorbar(im0, ax=axes[0, -1], fraction=0.04); plt.colorbar(im1, ax=axes[1, -1], fraction=0.04)
    plt.tight_layout(); plt.show()

    # ── Contactsheet: temporal module output, frame 1 ───────
    show_contactsheet(tm_out[0],
        title=f'Temporal Module [{mm_idx}] output — frame 1 — {tm_out.shape[1]} channels',
        ncols=min(tm_out.shape[1], 24), cmap='viridis', figsize=(18, 12))


### 9c — Scratch Layers (layer_rn)

Four 3×3 Conv2d layers without bias map all four feature levels to a **uniform 64-channel** representation. This is the 'scratch' network that prepares features for the hierarchical fusion.

| Layer | Input | Output |
|-------|-------|--------|
| layer1_rn | [T, 48,  148, 148] | [T, 64, 148, 148] |
| layer2_rn | [T, 96,   74,  74] | [T, 64,  74,  74] |
| layer3_rn | [T, 192,  37,  37] | [T, 64,  37,  37] |
| layer4_rn | [T, 384,  19,  19] | [T, 64,  19,  19] |


In [ ]:
scratch_names  = ['scratch_1','scratch_2','scratch_3','scratch_4']
resize_names   = ['resize_0', 'resize_1', 'resize_2', 'resize_2']  # inputs
# Note: scratch_3 and scratch_4 use temporal outputs
temporal_in    = [None, None, 'temporal_0', 'temporal_1']

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
fig.suptitle('Scratch layers — 3×3 Conv  →  uniform 64 channels', fontsize=11)

for col, (sn, prev) in enumerate(zip(scratch_names, ['resize_0','resize_1','temporal_0','temporal_1'])):
    inp = to_np(A[prev])   # [T, C_in, H, W]
    out = to_np(A[sn])     # [T, 64,   H, W]

    tinfo(torch.from_numpy(inp), f'{prev}  (input  to {sn})')
    tinfo(torch.from_numpy(out), f'{sn}  (output)')

    # mean |act| maps
    axes[0, col].imshow(norm01(np.abs(inp[0]).mean(0)), cmap='magma')
    axes[0, col].set_title(f'Input: {inp.shape[1]}ch {inp.shape[2]}×{inp.shape[3]}', fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(norm01(np.abs(out[0]).mean(0)), cmap='magma')
    axes[1, col].set_title(f'Output: {out.shape[1]}ch {out.shape[2]}×{out.shape[3]}', fontsize=8)
    axes[1, col].axis('off')

    # Full 64-ch contactsheet
    cs = make_contactsheet(out[0], ncols=8)
    axes[2, col].imshow(cs, cmap='viridis', aspect='auto')
    axes[2, col].set_title('All 64 channels (contactsheet)', fontsize=7)
    axes[2, col].axis('off')

plt.tight_layout(); plt.show()


### 9d — RefineNet4 (FeatureFusionBlock): Step-by-Step

`refinenet4` takes only **one input**: `layer_4_rn  [T, 64, 19, 19]`.

```
layer_4_rn  [T, 64, 19, 19]
  └─ resConfUnit2
       ├─ ReLU(x)
       ├─ conv1  3×3 → [T, 64, 19, 19]
       ├─ ReLU
       ├─ conv2  3×3 → [T, 64, 19, 19]
       └─ + skip (residual)
  └─ bilinear upsample to layer_3_rn.shape  (37, 37)
  └─ out_conv  1×1 → [T, 64, 37, 37]
  └─ temporal_module[2]  →  path_4  [T, 64, 37, 37]
```


In [ ]:
rn = 'refinenet4'
# Inputs / outputs we captured
in_rn4     = to_np(A['scratch_4'])             # [T, 64, 19, 19]
rcu2_out   = to_np(A[f'{rn}.rcu2'])            # [T, 64, 19, 19]  (after ResConfUnit2)
rn4_out    = to_np(A[rn])                      # [T, 64, 37, 37]  (after interp + out_conv)
rn4_oc_out = to_np(A[f'{rn}.out_conv'])        # [T, 64, 37, 37]
tm2_out    = unpack_temporal(A['temporal_2'])  # [B,C,T,H,W] → [T,64,H,W]  path_4 after temporal

# Inside rcu2
rcu2_c1 = to_np(A[f'{rn}.rcu2.conv1'])
rcu2_c2 = to_np(A[f'{rn}.rcu2.conv2'])

print("RefineNet4 tensor shapes:")
tinfo(torch.from_numpy(in_rn4),   '  input (scratch_4)       ')
tinfo(torch.from_numpy(rcu2_c1),  '  rcu2.conv1              ')
tinfo(torch.from_numpy(rcu2_c2),  '  rcu2.conv2              ')
tinfo(torch.from_numpy(rcu2_out), '  rcu2 output (after skip)')
tinfo(torch.from_numpy(rn4_oc_out),'  after bilinear+out_conv ')
tinfo(torch.from_numpy(tm2_out),  '  path_4 (after temporal2)')

fig, axes = plt.subplots(2, 6, figsize=(22, 7))
fig.suptitle('RefineNet4 — FeatureFusionBlock step-by-step  (mean |activation|, frame 1)', fontsize=11)

steps = [
    (in_rn4,    'Input\nlayer_4_rn\n[64,19×19]'),
    (rcu2_c1,   'rcu2.conv1\n(3×3 Conv)\n[64,19×19]'),
    (rcu2_c2,   'rcu2.conv2\n(3×3 Conv)\n[64,19×19]'),
    (rcu2_out,  'rcu2 output\n+ residual skip\n[64,19×19]'),
    (rn4_oc_out,'bilinear→37×37\n+ out_conv (1×1)\n[64,37×37]'),
    (tm2_out,   'path_4\n(after temporal_2)\n[64,37×37]'),
]
for col, (arr, label) in enumerate(steps):
    top = norm01(np.abs(arr[0]).mean(0))
    cs  = make_contactsheet(arr[0], ncols=8)
    axes[0, col].imshow(top, cmap='inferno')
    axes[0, col].set_title(label, fontsize=7)
    axes[0, col].axis('off')
    axes[1, col].imshow(cs, cmap='viridis', aspect='auto')
    axes[1, col].set_title('All 64 ch (contactsheet)', fontsize=7)
    axes[1, col].axis('off')

plt.tight_layout(); plt.show()


### 9e — RefineNet3: Fusing path_4 + layer_3_rn

`refinenet3` takes **two inputs**: `path_4 [T, 64, 37, 37]` and `layer_3_rn [T, 64, 37, 37]`.

```
xs = (path_4, layer_3_rn)
  ├─ resConfUnit1(layer_3_rn)   →  res  [T, 64, 37, 37]
  ├─ path_4 + res               →  fused [T, 64, 37, 37]
  ├─ resConfUnit2(fused)        →  [T, 64, 37, 37]
  ├─ bilinear upsample → layer_2_rn.shape  (74, 74)
  └─ out_conv  1×1              →  path_3  [T, 64, 74, 74]
  → temporal_module[3]          →  path_3 (temporal)  [T, 64, 74, 74]
```


In [ ]:
rn = 'refinenet3'
path4_in   = unpack_temporal(A['temporal_2'])   # [B,C,T,H,W] → [T,C,H,W]  path_4 after temporal_2
l3_rn      = to_np(A['scratch_3'])       # layer_3_rn [T, 64, 37, 37]
rcu1_out   = to_np(A[f'{rn}.rcu1'])      # resConfUnit1 applied to l3_rn
rcu2_out   = to_np(A[f'{rn}.rcu2'])      # resConfUnit2 on fused
rn3_oc     = to_np(A[f'{rn}.out_conv'])  # after interp+out_conv
tm3_out    = unpack_temporal(A['temporal_3'])   # [B,C,T,H,W] → [T,C,H,W]  path_3

print("RefineNet3 tensor shapes:")
tinfo(torch.from_numpy(path4_in),  '  path_4 input         ')
tinfo(torch.from_numpy(l3_rn),     '  layer_3_rn input     ')
tinfo(torch.from_numpy(rcu1_out),  '  rcu1(layer_3_rn)     ')
fused_np = path4_in + rcu1_out
tinfo(torch.from_numpy(fused_np),  '  fused (path_4+rcu1)  ')
tinfo(torch.from_numpy(rcu2_out),  '  rcu2 output          ')
tinfo(torch.from_numpy(rn3_oc),    '  after bilinear+out_conv')
tinfo(torch.from_numpy(tm3_out),   '  path_3 (after temporal3)')

fig, axes = plt.subplots(3, 6, figsize=(22, 10))
fig.suptitle('RefineNet3 — two-input FeatureFusionBlock  (frame 1)', fontsize=11)

row0 = [(path4_in, 'path_4\n[64,37×37]'),
        (l3_rn,    'layer_3_rn\n[64,37×37]'),
        (rcu1_out, 'rcu1(layer_3_rn)\n[64,37×37]'),
        (fused_np, 'path_4 + rcu1\n(fused, [64,37×37])'),
        (rcu2_out, 'rcu2(fused)\n[64,37×37]'),
        (rn3_oc,   'bilinear→74×74\n+out_conv\n[64,74×74]')]
row2 = [(tm3_out,  'path_3 after\ntemporal_3\n[64,74×74]')]

for col, (arr, label) in enumerate(row0):
    top = norm01(np.abs(arr[0]).mean(0))
    cs  = make_contactsheet(arr[0], ncols=8)
    axes[0, col].imshow(top, cmap='inferno')
    axes[0, col].set_title(label, fontsize=7)
    axes[0, col].axis('off')
    axes[1, col].imshow(cs, cmap='viridis', aspect='auto')
    axes[1, col].set_title('contactsheet', fontsize=7)
    axes[1, col].axis('off')

# Row 2: path_3 temporal
axes[2, 0].imshow(norm01(np.abs(tm3_out[0]).mean(0)), cmap='inferno')
axes[2, 0].set_title('path_3 after temporal_3 [64,74×74]', fontsize=7); axes[2,0].axis('off')
cs_tm3 = make_contactsheet(tm3_out[0], ncols=8)
axes[2, 1].imshow(cs_tm3, cmap='viridis', aspect='auto')
axes[2, 1].set_title('path_3 contactsheet', fontsize=7); axes[2, 1].axis('off')
for ax in axes[2, 2:]: ax.axis('off')
plt.tight_layout(); plt.show()


### 9f — RefineNet2 and RefineNet1

Same fusion pattern, progressing to finer scales and larger spatial resolution:

- **refinenet2**: `(path_3, layer_2_rn)` → `path_2  [T, 64, 148, 148]`
- **refinenet1**: `(path_2, layer_1_rn)` → `path_1  [T, 64, 296, 296]`  (scale_factor=2 upsample, no explicit size)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('RefineNet2 & RefineNet1 — hierarchical feature fusion', fontsize=11)

rn2_inp  = unpack_temporal(A['temporal_3'])   # [B,C,T,H,W] → [T,C,H,W]  path_3
l2_rn    = to_np(A['scratch_2'])        # [T, 64, 74, 74]
rn2_rcu1 = to_np(A['refinenet2.rcu1'])  # rcu1 on l2_rn
rn2_rcu2 = to_np(A['refinenet2.rcu2'])
rn2_out  = to_np(A['refinenet2'])       # path_2 [T,64,148,148]

rn1_inp  = rn2_out                      # path_2
l1_rn    = to_np(A['scratch_1'])        # [T, 64, 148, 148]
rn1_rcu1 = to_np(A['refinenet1.rcu1'])
rn1_rcu2 = to_np(A['refinenet1.rcu2'])
rn1_out  = to_np(A['refinenet1'])       # path_1 [T,64,296,296]

print("RefineNet2:")
tinfo(torch.from_numpy(rn2_inp), '  path_3 in       '); tinfo(torch.from_numpy(l2_rn),   '  layer_2_rn      ')
tinfo(torch.from_numpy(rn2_rcu1),'  rcu1(l2_rn)     '); tinfo(torch.from_numpy(rn2_rcu2),'  rcu2(fused)     ')
tinfo(torch.from_numpy(rn2_out), '  path_2 out      ')
print("RefineNet1:")
tinfo(torch.from_numpy(rn1_inp), '  path_2 in       '); tinfo(torch.from_numpy(l1_rn),   '  layer_1_rn      ')
tinfo(torch.from_numpy(rn1_rcu1),'  rcu1(l1_rn)     '); tinfo(torch.from_numpy(rn1_rcu2),'  rcu2(fused)     ')
tinfo(torch.from_numpy(rn1_out), '  path_1 out      ')

pairs = [
    ('RefineNet2 inputs', rn2_inp,  l2_rn,   rn2_out),
    ('RefineNet1 inputs', rn1_inp,  l1_rn,   rn1_out),
]
for row, (title, a, b, out) in enumerate(pairs):
    for col, (arr, label) in enumerate([(a, 'Primary input'), (b, 'Secondary input'), (out, 'Output')]):
        axes[row, col].imshow(norm01(np.abs(arr[0]).mean(0)), cmap='inferno')
        axes[row, col].set_title(f'{title}\n{label}  {list(arr.shape[1:])}', fontsize=8)
        axes[row, col].axis('off')
plt.tight_layout(); plt.show()

# ── Full contactsheets for path_2 and path_1 ──────────────────
for name, arr in [('path_2 (refinenet2 output)', rn2_out),
                  ('path_1 (refinenet1 output)', rn1_out)]:
    show_contactsheet(arr[0],
        title=f'{name}  {list(arr.shape[1:])}',
        ncols=8, cmap='viridis', figsize=(14, 9))


## 10 — Output Convolutions → Depth Map

```
path_1  [T, 64, 296, 296]
  └─ output_conv1   Conv2d(64, 32, 3×3)   →  [T, 32, 296, 296]
  └─ bilinear interp to (518, 518)         →  [T, 32, 518, 518]
  └─ output_conv2
        ├─ Conv2d(32, 32, 3×3)             →  [T, 32, 518, 518]
        ├─ ReLU
        ├─ Conv2d(32, 1, 1×1)              →  [T,  1, 518, 518]
        └─ ReLU
  └─ interpolate to original (H, W)        →  [T,  1, H, W]
  └─ ReLU + reshape                        →  [B, T, H, W]
```


In [ ]:
oc1 = to_np(A['out_conv1'])   # [T, 32, 296, 296]  (output_conv1 before interp)
oc2 = to_np(A['out_conv2'])   # [T, 1,  518, 518]  (full output_conv2 output)
depth = to_np(DEPTH_OUT[0])   # [T, H_orig, W_orig]

tinfo(torch.from_numpy(oc1),  'output_conv1  [T, 32, 296, 296]  ')
tinfo(torch.from_numpy(oc2),  'output_conv2  [T,  1, 518, 518]  ')
tinfo(torch.from_numpy(depth),'final depth   [T, H, W]           ')

# ── output_conv1: contactsheet of all 32 channels ─────────────
fig, axes = plt.subplots(1, T, figsize=(14, 4))
fig.suptitle('output_conv1  [32 channels, 296×296]  — mean |activation| per frame', fontsize=10)
for fr in range(T):
    axes[fr].imshow(norm01(np.abs(oc1[fr]).mean(0)), cmap='magma')
    axes[fr].set_title(f'Frame {fr+1}', fontsize=9)
    axes[fr].axis('off')
plt.tight_layout(); plt.show()

show_contactsheet(oc1[0], title='output_conv1 — all 32 channels (frame 1)',
                  ncols=8, cmap='plasma', figsize=(14, 5))

# ── Single-channel output_conv2 ───────────────────────────────
show_row(list(oc2[:, 0]),
         [f'output_conv2 (raw depth, 518×518) — Frame {i+1}' for i in range(T)],
         cmap='inferno', suptitle='Pre-final depth (before resize back to original resolution)')

# ── Final depth maps ───────────────────────────────────────────
vmin, vmax = depth.min(), depth.max()
show_row(list(depth),
         [f'Final depth — Frame {i+1}  [{H_in}×{W_in}]' for i in range(T)],
         cmap='inferno', vmin=vmin, vmax=vmax,
         suptitle='Final Depth Maps  (all frames, same colour scale)')

# ── Side-by-side: RGB vs depth for each frame ─────────────────
fig, axes = plt.subplots(2, T, figsize=(15, 8))
fig.suptitle('RGB vs Depth — side by side', fontsize=12)
for fr in range(T):
    axes[0, fr].imshow(RAW[fr])
    axes[0, fr].set_title(f'Frame {fr+1} — RGB', fontsize=9)
    axes[0, fr].axis('off')
    im = axes[1, fr].imshow(depth[fr], cmap='inferno', vmin=vmin, vmax=vmax)
    axes[1, fr].set_title(f'Frame {fr+1} — Depth', fontsize=9)
    axes[1, fr].axis('off')
plt.colorbar(im, ax=axes[1, -1], fraction=0.04, pad=0.02, label='relative depth')
plt.tight_layout(); plt.show()

print("\n=== Complete pipeline visualised ===")
print(f"Input:  3 × {H_in}×{W_in} RGB frames")
print(f"Output: 3 × {H_in}×{W_in} depth maps")


## Summary: Tensor Shape Journey

| Stage | Tensor | Shape |
|-------|--------|-------|
| Input | RGB frames | `[3, 480, 640, 3]` uint8 |
| After preprocessing | normalised CHW | `[3, 3, 518, 518]` float32 |
| patch_embed output | patch tokens | `[3, 1369, 384]` |
| + CLS + pos_embed | full sequence | `[3, 1370, 384]` |
| After each block | transformer tokens | `[3, 1370, 384]` |
| Extracted layers ×4 | patch tokens | `[3, 1369, 384]` each |
| After reshape | spatial | `[3, 384, 37, 37]` each |
| After projects[0] | projected | `[3, 48, 37, 37]` |
| After resize_layers[0] | ×4 upsample | `[3, 48, 148, 148]` |
| After projects[1]+resize[1] | ×2 upsample | `[3, 96, 74, 74]` |
| After projects[2]+resize[2] | identity | `[3, 192, 37, 37]` |
| After projects[3]+resize[3] | ÷2 downsample | `[3, 384, 19, 19]` |
| After temporal_0 (layer_3) | temporally refined | `[3, 192, 37, 37]` |
| After temporal_1 (layer_4) | temporally refined | `[3, 384, 19, 19]` |
| After scratch layer1_rn | uniform 64ch | `[3, 64, 148, 148]` |
| After scratch layer2_rn | uniform 64ch | `[3, 64, 74, 74]` |
| After scratch layer3_rn | uniform 64ch | `[3, 64, 37, 37]` |
| After scratch layer4_rn | uniform 64ch | `[3, 64, 19, 19]` |
| After refinenet4 | path_4 | `[3, 64, 37, 37]` |
| After temporal_2 | path_4 (temporal) | `[3, 64, 37, 37]` |
| After refinenet3 | path_3 | `[3, 64, 74, 74]` |
| After temporal_3 | path_3 (temporal) | `[3, 64, 74, 74]` |
| After refinenet2 | path_2 | `[3, 64, 148, 148]` |
| After refinenet1 | path_1 | `[3, 64, 296, 296]` |
| After output_conv1 | | `[3, 32, 296, 296]` |
| After interpolate | | `[3, 32, 518, 518]` |
| After output_conv2 | raw depth | `[3, 1, 518, 518]` |
| After final interp+relu | **depth output** | **`[1, 3, 518, 518]`** |
